# Raport: Lista 1

**Autor:** Bartosz Kruszewski
**Data:** 2026-10-09


In [ ]:
#| include: false

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from scipy import stats

In [ ]:
def calculate_metrics(estimates, true_val):
    bias = np.mean(estimates) - true_val
    var = np.var(estimates, ddof=1)
    mse = np.mean((estimates - true_val) ** 2)
    return bias, var, mse

In [ ]:
def est_mean(X):
    """Średnia arytmetyczna."""
    return np.mean(X, axis=1)


def est_median(X):
    """Mediana."""
    return np.median(X, axis=1)


def est_weighted_custom(X):
    """Średnia ważona z własnymi wagami trójkątnymi."""
    n = X.shape[1]

    # wyliczenie wag trójkątnych
    w = np.arange(1, n + 1, dtype=float)
    w = np.minimum(w, n + 1 - w)
    weights = w / np.sum(w)

    return np.dot(X, weights)


def est_order_stats(X):
    """Średnia ważona statystyk pozycyjnych oparta na N(0,1)."""

    n = X.shape[1]
    X_sorted = np.sort(X, axis=1)

    # wyliczenie wag na podstawie kwantyli rozkładu normalnego
    p = (np.arange(1, n + 1) - 0.5) / n
    z = stats.norm.ppf(p)
    w = stats.norm.pdf(z)
    weights = w / np.sum(w)

    return np.dot(X_sorted, weights)


def est_harmonic_mean(X):
    """Średnia harmoniczna: n / sum(1/X_i)."""
    n = X.shape[1]
    with np.errstate(divide="ignore", invalid="ignore"):
        inv_sum = np.sum(1.0 / X, axis=1)
        res = n / inv_sum
    return res


def est_power_mean_3(X):
    """Średnia potęgowa rzędu 3."""
    mean_cube = np.mean(X**3, axis=1)
    return np.sign(mean_cube) * np.abs(mean_cube) ** (1.0 / 3.0)


def est_trimmed_mean(X, trim_ratio=0.10):
    """Estymator własny: średnia ucinana."""
    n = X.shape[1]
    k = int(trim_ratio * n)
    if k == 0:
        return np.mean(X, axis=1)
    X_sorted = np.sort(X, axis=1)
    return np.mean(X_sorted[:, k : n - k], axis=1)


def est_sample_variance(X):
    """Estymator wariancji S^2."""
    return np.var(X, axis=1, ddof=1)


def est_variance_around_median(X):
    """Estymator wariancji wokół mediany."""
    n = X.shape[1]
    med = np.median(X, axis=1, keepdims=True)
    return np.sum((X - med) ** 2, axis=1) / (n - 1)


def est_median_squared_dev(X):
    """Mediana kwadratów odchyleń od średniej."""
    mean = np.mean(X, axis=1, keepdims=True)
    sq_deviations = (X - mean) ** 2
    return np.median(sq_deviations, axis=1)

In [ ]:
#| include: false


def run_simulation(n, loc, scale, theta, dist_func, estimators, M=10000):
    X = dist_func(loc=loc, scale=scale, size=(M, n))
    estimated = {est.__name__[4:]: est(X) for est in estimators}

    metrics = pd.DataFrame(
        [(name, *calculate_metrics(vals, theta)) for name, vals in estimated.items()],
        columns=pd.Index(["Estymator", "Obciążenie", "Wariancja", "MSE"]),
    )

    title = f"Metryki estymatorów (n={n}, loc={loc}, scale={scale}, theta={theta})"

    display(metrics.style.set_caption(title).hide(axis="index").format(precision=5))

    g = sns.boxplot(data=estimated, showmeans=True)
    g.axhline(theta, color="red", ls="--", label=f"theta={theta}")
    g.set(title=title, xlabel="Estymator", ylabel="Wartość estymatora")
    g.tick_params(axis="x", rotation=15)
    g.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
#| echo: false
#| output: true

for n in [20, 100]:
    for theta in [0, 4]:
        for sigma in [1, 2]:
            run_simulation(
                n=n,
                loc=theta,
                scale=sigma,
                theta=theta,
                dist_func=np.random.normal,
                estimators=[
                    est_mean,
                    est_median,
                    est_weighted_custom,
                    est_order_stats,
                    est_harmonic_mean,
                    est_power_mean_3,
                    est_trimmed_mean,
                ],
            )

In [ ]:
#| echo: false
#| output: true

for dist_func in [np.random.normal, np.random.logistic, stats.cauchy.rvs]:
    for theta in [0, 4]:
        for sigma in [1, 2]:
            run_simulation(
                n=50,
                loc=theta,
                scale=sigma,
                theta=theta,
                dist_func=dist_func,
                estimators=[est_mean, est_median],
            )

In [ ]:
#| echo: false
#| output: true

M, N = 10000, 50

metrics = pd.DataFrame(
    [
        {"k": k, "Estymator": est.__name__[4:], "Metryka": m, "Wartość": val}
        for k in range(10, 101, 10)
        for X in [np.hstack((np.random.normal(size=(M, N - 1)), np.full((M, 1), k)))]
        for est in [est_mean, est_median]
        for m, val in zip(
            ["Obciążenie", "Wariancja", "MSE"], calculate_metrics(est(X), theta)
        )
    ]
)

g = sns.relplot(
    data=metrics,
    x="k",
    y="Wartość",
    hue="Estymator",
    style="Estymator",
    col="Metryka",
    kind="line",
    markers=True,
    dashes=False,
    facet_kws={"sharey": False},
)
g.set_axis_labels("Wartość k", "Wartość metryki")
g.set_titles(col_template="{col_name}")
plt.show()

display(
    metrics.pivot(
        index="k", columns=["Metryka", "Estymator"], values="Wartość"
    ).style.format(precision=5)
)

In [ ]:
#| echo: false
#| output: true

for mu in [0, 4]:
    for theta in [1, 4]:
        run_simulation(
            n=50,
            loc=mu,
            scale=np.sqrt(theta),
            theta=theta,
            dist_func=np.random.normal,
            estimators=[
                est_sample_variance,
                est_variance_around_median,
                est_median_squared_dev,
            ],
        )